# Phase 1 - Original Dataset Inventory in MinIO

The old progress-review fixture downloaded only 72 JFK rows. The final demo uses the original large datasets already imported into MinIO:

- ARCO-ERA5 airport-hour weather for 2015-2024.
- BTS Reporting Carrier On-Time Performance ZIP archives for 2015-2024.
- OurAirports metadata used to map flight origins to ERA5 airport keys.

The separate root-level `ARCO_ERA5_Dataset_Download_Script.ipynb` documents the one-time Google Colab extraction workflow.

In [1]:
import os
from collections import defaultdict
import boto3
import pandas as pd
from IPython.display import display

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT_INTERNAL"],
    aws_access_key_id=os.environ["MINIO_ROOT_USER"],
    aws_secret_access_key=os.environ["MINIO_ROOT_PASSWORD"],
    region_name=os.getenv("AWS_REGION", "us-east-1"),
)
RAW_BUCKET = os.environ["RAW_BUCKET"]

In [2]:
def list_objects(prefix):
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=RAW_BUCKET, Prefix=prefix):
        yield from page.get("Contents", [])

def object_summary(prefix):
    objects = list(list_objects(prefix))
    return {
        "prefix": prefix,
        "objects": len(objects),
        "size_gib": round(sum(obj["Size"] for obj in objects) / 1024**3, 3),
    }

inventory = pd.DataFrame([
    object_summary("arco_era5_us_airport_hourly/"),
    object_summary("bts_on_time/raw_zip/"),
    object_summary("metadata/"),
])
display(inventory)

,prefix,objects,size_gib
0,arco_era5_us_airport_hourly/,840,44.445
1,bts_on_time/raw_zip/,120,2.971
2,metadata/,133,0.027


## Verify the Full Weather Layout

The extraction workflow writes six Parquet airport shards and one `_SUCCESS.json` marker for every monthly partition.

In [3]:
weather_objects = list(list_objects("arco_era5_us_airport_hourly/"))
weather_keys = [obj["Key"] for obj in weather_objects]
parquet_keys = [key for key in weather_keys if key.endswith(".parquet")]
success_keys = [key for key in weather_keys if key.endswith("_SUCCESS.json")]
years = sorted({part.split("=")[1] for key in parquet_keys for part in key.split("/") if part.startswith("year=")})

print("Weather Parquet files:", len(parquet_keys))
print("Monthly success markers:", len(success_keys))
print("Years:", years)
assert len(parquet_keys) == 720
assert len(success_keys) == 120
assert years == [str(year) for year in range(2015, 2025)]

Weather Parquet files: 720
Monthly success markers: 120
Years: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


## Verify BTS Archives

BTS provides the actual cancellation and arrival-delay outcomes used by the Silver labels. These labels replace the old weather-rule proxy.

In [4]:
bts_keys = [obj["Key"] for obj in list_objects("bts_on_time/raw_zip/") if obj["Key"].endswith(".zip")]
print("BTS monthly ZIP files:", len(bts_keys))
print("First archive:", sorted(bts_keys)[0])
print("Last archive:", sorted(bts_keys)[-1])
assert len(bts_keys) == 120
print("Original-data inventory validation complete.")

BTS monthly ZIP files: 120
First archive: bts_on_time/raw_zip/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_1.zip
Last archive: bts_on_time/raw_zip/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_9.zip
Original-data inventory validation complete.
